# LLM Alignment via Direct Preference Optimization (DPO) on NVIDIA Tesla T4 (16GB)

This interactive notebook demonstrates the complete end-to-end workflow for aligning a large language model using **Direct Preference Optimization (DPO)** on a single **NVIDIA Tesla T4 (16GB VRAM)** GPU.

### Pipeline Overview
1. **Environment & Hardware Diagnostics**: Inspecting GPU compute capability (Turing CC 7.5) and precision setup.
2. **Dataset Inspection & Analysis**: Validating pairwise preference schema (`prompt`, `chosen`, `rejected`) and length bias.
3. **Memory Budget Engineering**: Modeling VRAM allocations for 4-bit QLoRA, LoRA adapters, and DPO dual forward passes.
4. **Model Initialization & QLoRA Configuration**: Loading base model with `bitsandbytes` NF4 and PEFT LoRA.
5. **DPO Training Loop**: Executing preference optimization with HuggingFace TRL / Soup.
6. **Evaluation & Before/After Comparison**: Computing implicit rewards $r_\theta(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$, margins, and qualitative generations.
7. **Model Merging & Export**: Packaging standalone 16-bit merged weights for production deployment.

## 1. Environment & Hardware Diagnostics

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")

## 2. Dataset Inspection & Token Analysis

In [ ]:
import json
from pathlib import Path

train_file = Path("./data/train.jsonl")
eval_file = Path("./data/eval.jsonl")

train_samples = [json.loads(l) for l in train_file.read_text(encoding='utf-8').strip().split('\n') if l.strip()]
eval_samples = [json.loads(l) for l in eval_file.read_text(encoding='utf-8').strip().split('\n') if l.strip()]

print(f"Loaded {len(train_samples)} training pairs and {len(eval_samples)} evaluation pairs.")
print("\n--- Sample Pair 1 ---")
print(f"Prompt:   {train_samples[0]['prompt']}")
print(f"Chosen:   {train_samples[0]['chosen'][:100]}...")
print(f"Rejected: {train_samples[0]['rejected'][:100]}...")

In [ ]:
# Run dataset inspection utility script
!python ./scripts/inspect_dataset.py --data-dir ./data

## 3. Memory Budget Calculation for NVIDIA T4 (16GB)

In [ ]:
# Run memory budget calculator
!python ./scripts/memory_budget.py --model-params 1.54 --model-name Qwen2.5-1.5B-Instruct --quantization 4bit --lora-rank 16 --seq-len 1024 --batch-size 1 --matrix

## 4. Pre-Flight Verification & Dry Run

In [ ]:
# Run pre-flight check suite
!python ./scripts/verify_training.py --config ./configs/dpo_t4.yaml

## 5. DPO Fine-Tuning Execution

In [ ]:
"""
DPO Training Script using HuggingFace TRL / PyTorch
"""
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import yaml

# Load configuration
with open('./configs/dpo_t4.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("DPO Configuration Loaded:")
print(yaml.dump(config, default_flow_style=False))

# In full training mode on GPU, soup can be invoked directly:
# !soup train --config ./configs/dpo_t4.yaml
print("Ready for training execution.")

## 6. Evaluation & Before vs After Benchmark

In [ ]:
# Run evaluation comparison script
!python ./scripts/evaluate_before_after.py --eval-file ./data/eval.jsonl --beta 0.1

## 7. Model Merging & Export for Production

In [ ]:
"""
Export & Merge LoRA Adapter Weights to Base Model
"""
def merge_and_export(base_model_id: str, adapter_dir: str, output_dir: str):
    print(f"Merging adapter '{adapter_dir}' into '{base_model_id}'...")
    # Code for merging weights:
    # from peft import PeftModel
    # base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.float16, device_map='auto')
    # peft_model = PeftModel.from_pretrained(base_model, adapter_dir)
    # merged = peft_model.merge_and_unload()
    # merged.save_pretrained(output_dir)
    print(f"Merged model saved to: {output_dir}")

merge_and_export(
    base_model_id="Qwen/Qwen2.5-1.5B-Instruct",
    adapter_dir="./output/dpo_t4_model",
    output_dir="./output/dpo_t4_merged_16bit"
)